# ML-08: Capstone Modeling Lane

Week 5. Lane 2, same slice as weeks 3 and 4: pages at the 1 April 2026 decision date, features from March only, label = declined in April by more than 20% and at least 20 points worse than the page's own client. The week 4 rule (`sliding_and_under_clicked`) is frozen and is the thing to beat, on the same rows, the same split and the same metrics. June stays sealed for week 6.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

def find_repo_root(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / "work" / "scripts" / "warehouse.py").exists():
            return folder
    raise FileNotFoundError("Run this from inside a clone of the repo.")

ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "work" / "scripts"))
import warehouse as W  # shared loader + Lane 2 frame builder (weeks 3-6)

con = W.connect()

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
frame = W.build_frame(con, feature_month="2026-03", label_month="2026-04")
y = frame["label"].to_numpy()
groups = frame["client_hash_id"]
print(f"pages: {len(frame):,}   clients: {groups.nunique()}   base rate: {y.mean():.1%}")

pages: 84,347   clients: 36   base rate: 27.0%


## 1. Method choice and why

The question is "which pages first?", so the output is a ranking, and every method below is judged on the ranking it produces, not on accuracy at a 0.5 threshold. I'm comparing three classifiers whose predicted probability becomes the score:

- **Logistic regression**: the readable one. If it matches the trees, I'd ship it, because a reviewer can see exactly how each signal pushes the score.
- **Random forest**: handles the interactions week 2 found (the same signal pointing different ways for different page ages) without me writing them by hand. Leaves of at least 50 pages keep it from memorising single pages.
- **Gradient boosting** (`HistGradientBoostingClassifier`): usually the strongest on tabular data. It's here to check whether extra power buys anything.

Features: the five from the week 3 contract, plus three that week 4 earned:

| feature | from | why it's in |
|---|---|---|
| `log_imp`, `days_visible`, `avg_position`, `momentum`, `content_age_days` | March / creation date | the week 3 contract |
| `ctr_pct` | March clicks / impressions | under-clicking was CONFIRMED in week 4 |
| `ctr_gap` | CTR minus its position tier's median, medians fitted on training clients only | the week 4 rule's own signal, made leak-free across folds |
| `client_momentum` | median March momentum of the page's own client | the label is client-relative, so the model needs to know what the client was doing |

Search volume stays out as a risk signal (OPPOSITE in week 4). It comes back only as a demand weight in section 3. `word_count` and the update dates stay out: they're July snapshots that can postdate the decision (week 3).

In [2]:
FEATURES = W.FIVE_FEATURES + ["ctr_pct", "ctr_gap", "client_momentum"]

frame["tier"] = pd.cut(frame["avg_position"], [0, 3, 10, 20, 50, np.inf], labels=False, include_lowest=True)
frame["client_momentum"] = frame.groupby("client_hash_id")["momentum"].transform("median")

def make_models():
    return {
        "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
        "random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=50, n_jobs=-1, random_state=SEED),
        "gradient boosting": HistGradientBoostingClassifier(random_state=SEED),
    }

def frozen_baseline(df):
    """The week 4 rule exactly as written (tier medians from the whole March frame)."""
    med = df.groupby("tier")["ctr_pct"].transform("median")
    flagged = df["momentum"].lt(-0.1) & (df["ctr_pct"] - med).lt(0)
    return np.where(flagged, -df["momentum"], 0.0)

baseline = frozen_baseline(frame)
print(f"baseline precision@50 on this frame: {W.precision_at_k(baseline, y, 50):.2f}  (week 4 receipt: 0.96)")

baseline precision@50 on this frame: 0.96  (week 4 receipt: 0.96)


## 2. Split design

Grouped 5-fold cross-validation by client (`GroupKFold` on `client_hash_id`). Every page is scored by a model that never saw any page from its client.

Why grouped: pages from the same client share a site, a tracking setup and, very often, one site-wide event. Week 3 showed client median changes from -87% to +96% in a single month. A random row split would let the model learn "this client is crashing" from training pages and then get credit for it on test pages of the same client. The honest question for a review queue is whether it works on an account it hasn't seen.

Why not time-aware yet: there's one decision date here, so time can't be split inside it. Week 6 trains on earlier months and tests on later ones, then opens the sealed June month once.

The out-of-fold scores from all five folds are pooled into one ranking over all 84,347 pages, so the model and the frozen rule are compared on exactly the same rows. Per-fold numbers are shown too, because two folds are a single large client each.

In [3]:
folds = list(GroupKFold(n_splits=5).split(frame, y, groups))
oof = {name: np.zeros(len(frame)) for name in make_models()}
fold_rows = []

for i, (tr, te) in enumerate(folds):
    # CTR gap: tier medians learned on training clients only, applied to both sides.
    med = frame.iloc[tr].groupby("tier")["ctr_pct"].median()
    frame["ctr_gap"] = frame["ctr_pct"] - frame["tier"].map(med)
    X = frame[FEATURES]
    row = {"fold": i, "test clients": groups.iloc[te].nunique(), "test pages": len(te),
           "base rate": round(y[te].mean(), 3),
           "baseline P@50": W.precision_at_k(baseline[te], y[te], 50)}
    for name, model in make_models().items():
        model.fit(X.iloc[tr], y[tr])
        oof[name][te] = model.predict_proba(X.iloc[te])[:, 1]
        row[f"{name} P@50"] = W.precision_at_k(oof[name][te], y[te], 50)
    fold_rows.append(row)

pd.DataFrame(fold_rows)

,fold,test clients,test pages,base rate,baseline P@50,logistic regression P@50,random forest P@50,gradient boosting P@50
0,0,1,20970,0.314,0.98,0.94,0.96,0.96
1,1,1,16337,0.263,0.96,0.96,0.96,0.88
2,2,11,15680,0.227,0.90,0.82,0.88,0.90
3,3,13,15680,0.278,0.80,0.76,0.86,0.74
4,4,10,15680,0.252,0.86,0.90,0.86,0.76


## 3. Train + compare vs my baseline

One table, pooled out-of-fold, same rows for everything. The last two rows multiply a score by the page's March impressions per day, which turns "chance of decline" into "expected demand lost". That's where volume belongs, after week 4 showed it's the opposite of a risk signal.

In [4]:
import json

lost = frame["lost_per_day"].to_numpy()
demand = (frame["imp"] / 31).to_numpy()
rng = np.random.default_rng(SEED)

scores = {
    "random": rng.random(len(frame)),
    "baseline rule (week 4)": baseline,
    **oof,
    "baseline rule x demand": baseline * demand,
    "random forest x demand": oof["random forest"] * demand,
}

def evaluate(s):
    return {
        "ROC AUC": roc_auc_score(y, s),
        "avg precision": average_precision_score(y, s),
        "P@20": W.precision_at_k(s, y, 20),
        "P@50": W.precision_at_k(s, y, 50),
        "P@500": W.precision_at_k(s, y, 500),
        "demand captured@50": W.demand_captured_at_k(s, y, lost, 50),
        "demand captured@500": W.demand_captured_at_k(s, y, lost, 500),
    }

table = pd.DataFrame({name: evaluate(s) for name, s in scores.items()}).T.round(3)
print(f"base rate: {y.mean():.3f}   rows: {len(y):,}   split: GroupKFold(5) by client, pooled out-of-fold")
(ROOT / "work" / "outputs" / "w05_model_metrics.json").write_text(json.dumps(
    {"split": "GroupKFold(5) by client_hash_id, pooled OOF", "decision_date": "2026-04-01",
     "base_rate": round(float(y.mean()), 4), "rows": int(len(y)), "seed": SEED,
     "table": table.to_dict(orient="index")}, indent=2))
table

base rate: 0.270   rows: 84,347   split: GroupKFold(5) by client, pooled out-of-fold


,ROC AUC,avg precision,P@20,P@50,P@500,demand captured@50,demand captured@500
random,0.502,0.272,0.30,0.24,0.268,0.006,0.024
baseline rule (week 4),0.595,0.389,1.00,0.96,0.874,0.069,0.070
logistic regression,0.692,0.480,0.90,0.88,0.806,0.059,0.035
random forest,0.706,0.497,0.95,0.94,0.820,0.007,0.030
gradient boosting,0.693,0.477,0.85,0.88,0.798,0.044,0.036
baseline rule x demand,0.592,0.348,0.70,0.70,0.568,0.290,0.315
random forest x demand,0.521,0.287,0.40,0.48,0.378,0.508,0.509


The honest reading has two halves.

The models rank the whole list better. Random forest has the best ROC AUC (0.706 against the rule's 0.595) and the best average precision (0.497 against 0.389), and all three models beat the rule on both. That's the part of the list the rule can't see: it gives 83% of pages a flat zero, so it can't order them at all.

At the top, the rule wins or ties. It scores 1.00 at 20, 0.96 at 50 and 0.874 at 500; the forest scores 0.95, 0.94 and 0.820. Week 4 showed why the rule's top is so strong: pages that dropped out of search in late March almost always stay out, and the rule ranks exactly those first. The models don't add anything there. Gradient boosting, the most flexible of the three, isn't better than the forest on any column, so the extra complexity doesn't earn its place.

Demand is where the choice becomes real. Unweighted, every method recovers under 7% of the possible lost demand in its top 50. Weighted by March demand, the forest captures 0.508 of it at 50 against the rule's 0.290, but its precision drops to 0.48 while the weighted rule keeps 0.70. Put plainly: rule x demand hands a reviewer more correct picks, forest x demand hands them fewer correct picks that are much bigger.

Per fold, top-50 precision swings from 0.74 to 0.98 depending on which clients are held out, so a gap of a few points at K = 50 isn't meaningful here.

What I'd use: the rule (or the forest; they tie) for a 50-page precision queue, and forest x demand when the goal is protecting the most traffic. Neither wins outright, and I'm not going to claim the model does.

## 4. Errors and interpretation

What the forest leans on, measured by shuffling one feature at a time on held-out clients and watching average precision fall (permutation importance, fold 2, which mixes 11 clients).

In [5]:
from sklearn.inspection import permutation_importance

tr, te = folds[2]
med = frame.iloc[tr].groupby("tier")["ctr_pct"].median()
frame["ctr_gap"] = frame["ctr_pct"] - frame["tier"].map(med)
rf = make_models()["random forest"].fit(frame.iloc[tr][FEATURES], y[tr])
imp = permutation_importance(rf, frame.iloc[te][FEATURES], y[te], scoring="average_precision",
                             n_repeats=5, random_state=SEED, n_jobs=1)
pd.DataFrame({"AP drop when shuffled": imp.importances_mean, "spread": imp.importances_std},
             index=FEATURES).sort_values("AP drop when shuffled", ascending=False).round(4)

,AP drop when shuffled,spread
momentum,0.1983,0.0031
content_age_days,0.0246,0.0009
ctr_pct,0.0121,0.0029
days_visible,0.0007,0.0013
avg_position,-0.0010,0.0015
ctr_gap,-0.0011,0.0033
log_imp,-0.0019,0.0010
client_momentum,-0.0170,0.0032


In [6]:
# The readable model: standardized logistic-regression coefficients (same fold).
lr = make_models()["logistic regression"].fit(frame.iloc[tr][FEATURES], y[tr])
coef = pd.Series(lr[-1].coef_[0], index=FEATURES).sort_values()
coef.round(3).to_frame("coefficient (per 1 std)")

,coefficient (per 1 std)
ctr_gap,-1.326
momentum,-0.744
log_imp,-0.211
content_age_days,-0.179
days_visible,-0.126
client_momentum,0.169
avg_position,0.172
ctr_pct,0.901


Momentum carries the forest. Shuffling it drops average precision by about 0.20, roughly eight times more than the next feature (content age, 0.025). CTR adds a little. Four features add nothing measurable on held-out clients, and one does harm: shuffling `client_momentum` *raises* average precision by 0.017. A client-level number is identical for every page of that client, so on training clients the forest can use it as a stand-in for "which client is this", and that doesn't carry over to new clients. It's the first thing the week 6 audit removes.

The logistic coefficients agree on direction once the collinear pair is read together. `ctr_gap` (-1.33) and `ctr_pct` (+0.90) are nearly the same column minus a tier constant, so their net effect is that a CTR below what the tier usually earns raises the chance of decline, which matches week 4. A sliding late March raises it too (-0.74 per standard deviation of momentum), and older and bigger pages are slightly safer, again in line with the week 4 bucket tables.

In [7]:
frame["rf_score"] = oof["random forest"]
frame["baseline_score"] = baseline
ranked = frame.sort_values("rf_score", ascending=False).reset_index(drop=True)
top = ranked.head(500)

show = ["content_hash_id", "rf_score", "imp", "days_visible", "avg_position", "momentum", "ctr_pct",
        "client_momentum", "change_pct", "client_median_change", "label"]

print("Where the forest is wrong at the top: false positives in its top 500, by client")
fp = top[top["label"] == 0]
print(f"{len(fp)} false positives; share from the single most common client: "
      f"{fp['client_hash_id'].value_counts(normalize=True).iloc[0]:.0%}")
print(f"median April change of those pages: {fp['change_pct'].median():.1f}%  "
      f"(their clients' median: {fp['client_median_change'].median():.1f}%)")
fp[show].head(3).round(3)

Where the forest is wrong at the top: false positives in its top 500, by client
90 false positives; share from the single most common client: 20%
median April change of those pages: -0.6%  (their clients' median: -34.4%)


,content_hash_id,rf_score,imp,days_visible,avg_position,momentum,ctr_pct,client_momentum,change_pct,client_median_change,label
11,content_4e19890a6d969be6,0.846,410.0,20,3.102,-2.485,0.0,0.010,24.252,-5.307,0
25,content_253493466ce9e161,0.839,144.0,10,5.590,-2.361,0.0,0.030,-38.287,-57.693,0
30,content_509d80e0ae1812ea,0.837,141.0,17,4.156,-1.984,0.0,-0.084,25.319,-18.017,0


In [8]:
# The costly misses: real declines with the most demand lost that the forest ranked outside its top 5,000.
missed = ranked.iloc[5000:]
missed = missed[missed["label"] == 1].sort_values("lost_per_day", ascending=False)
print(f"declines ranked outside the top 5,000: {len(missed):,} of {y.sum():,}")
missed[show + ["lost_per_day"]].head(3).round(3)

declines ranked outside the top 5,000: 19,305 of 22,768


,content_hash_id,rf_score,imp,days_visible,avg_position,momentum,ctr_pct,client_momentum,change_pct,client_median_change,label,lost_per_day
24623,content_44f34c0a90047651,0.297,212404.0,31,0.666,3.280,0.011,-0.084,-75.512,-18.017,1,5173.909
79817,content_66288edeb93b7c4f,0.070,137878.0,31,13.584,1.733,0.567,-0.084,-90.058,-18.017,1,4005.477
23048,content_34a70fea29d15f24,0.307,143019.0,31,3.166,-0.124,0.030,-0.069,-79.569,-45.774,1,3670.916


In [9]:
# Does the model agree with the rule? Overlap of the two top-500 lists.
rule_top = set(frame.sort_values("baseline_score", ascending=False).head(500)["content_hash_id"])
rf_top = set(top["content_hash_id"])
both = rule_top & rf_top
only_rf = frame[frame["content_hash_id"].isin(rf_top - rule_top)]
print(f"pages in both top-500 lists: {len(both)}")
print(f"forest-only picks: {len(only_rf)}, precision {only_rf['label'].mean():.2f}, "
      f"median March impressions {only_rf['imp'].median():.0f}, "
      f"share flagged by the rule {(only_rf['baseline_score'] > 0).mean():.0%}")

pages in both top-500 lists: 214
forest-only picks: 286, precision 0.78, median March impressions 252, share flagged by the rule 97%


Three kinds of error, each with a different cause.

**False positives at the top are mostly "dropped and came back".** The forest's top 500 holds 90 pages that didn't decline. Their median April change was -0.6% while their clients' median was -34.4%, so they held up better than their own sites. The three shown fell hard in late March (momentum around -2 to -2.5, visible on 10 to 20 days) and then recovered or held in April. A late-month dip can be a temporary indexing wobble, and nothing in March tells the two apart.

**The costly misses are big pages that fell with no warning.** The three largest misses had 138K to 212K March impressions, were visible every day of March, and were flat or rising in late March, then fell 75% to 90% in April. A March-only model has no signal for a drop like that. It's the main limit of a one-month feature window and the reason the capstone should try longer windows. More broadly, 19,305 of the 22,768 declines sit outside the forest's top 5,000, so most declines can't be reached at review-queue depth by any of these methods.

**The forest mostly re-orders the rule's picks.** 97% of the forest-only top-500 pages are flagged by the rule anyway. The model gives a sharper ordering inside the rule's flagged set and an ordering for pages the rule scores zero; it isn't a different idea of what a declining page looks like.

## Self-check

- [x] Every section above is filled, markdown thinking and the code that backs it
- [x] The notebook runs top to bottom with no errors (seed 42 throughout)
- [x] No client names, URLs, or private queries anywhere (pseudonymized hashes and aggregates only)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, metrics receipt in `work/outputs/w05_model_metrics.json`